In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

In [ ]:
# Loud dataset
df = pd.read_csv("job_salary_prediction_dataset.csv")
df.head()

In [ ]:
#Check Data Types
df.dtypes

In [ ]:
# List of nominal categorical columns
nominal_cols = ['job_title', 'industry', 'location', 'remote_work']

# Convert to category type
for col in nominal_cols:
    df[col] = df[col].astype('category')

print("Nominal columns converted to 'category'.")

df.dtypes

In [ ]:
# Ordinal Encoding (Ranking)
# Define the order for Education
edu_map = {
    'High School': 0,
    'Diploma': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4
}

# Define the order for Company Size
size_map = {
    'Startup': 0,
    'Small': 1,
    'Medium': 2,
    'Large': 3,
    'Enterprise': 4
}

# Apply the mapping
df['education_level_rank'] = df['education_level'].map(edu_map)
df['company_size_rank'] = df['company_size'].map(size_map)

# results
df[['education_level', 'education_level_rank', 'company_size', 'company_size_rank']].head()

In [ ]:
# Handling Missing Values
# Detect Missing Values
df.isna()

In [ ]:
df.isna().sum()

In [ ]:
df2 = df.copy()
df2.loc[0:5, 'salary'] = np.nan

In [ ]:
df2.isna().sum()

In [ ]:
df2.head(10)

In [ ]:
#Strategy for missing values: Median Imputation
df_median = df2.copy()
df_median['salary'].fillna(df_median['salary'].median(), inplace=True)

The strategy chosen for this dataset is Median Imputation.
Salary data is almost always "skewed." In any large company or industry, most people earn a certain range, while a few (outliers) earn significantly more.
Using the median ensures that the missing values which will be filled are realistic and don't accidentally "inflate" the data, which helps the future machine learning model stay accurate.

In [ ]:
df_median.head(10)

In [ ]:
# Handling Outliers
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df['salary'])

plt.title('Boxplt of Salary')
plt.show()

In [ ]:
# Detect Outliers using IQR
Q1 = df['salary'].quantile(0.25)
Q3 = df['salary'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['salary'] < lower) | (df['salary'] > upper)]
outliers.head(15)

In [ ]:
# Capping Outliers
lower_cap = df['salary'].quantile(0.05) # 5%
upper_cap = df['salary'].quantile(0.95) # 95%

df_capped = df.copy()
df_capped['salary'] = df_capped['salary'].clip(lower_cap, upper_cap)

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df_capped['salary'])

plt.title('Boxplt of Salary')
plt.show()

In [ ]:
# Data Transformation – Normalization
df[['experience_years','skills_count', 'certifications', 'salary']].head()

In [ ]:
# Min-Max Normalization
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = df[['experience_years','skills_count', 'certifications', 'salary']].copy()

df_scaled[['experience_years','skills_count', 'certifications', 'salary']] = scaler.fit_transform(df_scaled)

df_scaled.head()

In [ ]:
# Z-Score Normalization
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_standardized = df[['experience_years','skills_count', 'certifications', 'salary']].copy()

df_standardized[['experience_years','skills_count', 'certifications', 'salary']] = scaler.fit_transform(df_standardized)

df_standardized.head()

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(df_standardized[['experience_years','skills_count', 'certifications', 'salary']].corr(), 
            annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap (Before PCA)")
plt.show()


Upon inspecting the correlation heatmap,
the Pearson correlation coefficients between numerical features are near zero ($|r| \approx 0.01$).
Since PCA relies on linear relationships and redundancy between variables to reduce dimensionality,
applying it to this uncorrelated dataset would not result in meaningful feature compression. Therefore, PCA was not performed.